# CASIA-FASD train + test: C / P3-SF / R7-SC — 100K and full-scale

Evaluation only: all extracted train and test **color** frames from `minhnh2107/casiafasd`.
Six frozen CelebA-trained checkpoints, no CASIA training, adaptation, threshold fitting or checkpoint selection.

**Kaggle inputs**
1. CASIA dataset used by `04_casia_fasd_dataset_explorer.ipynb`.
2. Upload the complete `runs/` directory from experiment 17 (100K).
3. Upload experiment 18 (full-scale), including its `runs/` directory.
4. Mount the SCRFD dataset containing `det_500m.onnx` (same detector as resources).

Keep each run's `best.pth`, `config.json` and `evaluation_freeze.json` (or `completion.json`). No historical resources, contract MD, or README sources are needed.
MiniFASNet architecture and exact crop helper are embedded below. Names of outer upload folders, including `_100k` / `100k` and duplicate suffixes, do not matter; discovery uses the six **run IDs**.

Enable GPU and Internet for the first dependency setup. Set paths only if auto-discovery is ambiguous.
`RUN_FILTER=None` runs all six; use a list to run any subset in parallel Kaggle notebooks.

**Protocol:** identical shared SCRFD detections → exact crop1.5 → 80×80 BGR float [0,1].
Binary score = logit(real) − logit(attack). Each run keeps its own frozen source threshold.
Results are reported separately for **train**, **test** and **combined**.
Combined deduplicates reconstructed `(video_id, frame_index)` identities before detection, preferring the test copy; train/test metrics retain all images in their own split. This is identity-key deduplication, not a claim of byte-identical JPEGs.
Primary: mean-score reconstructed-video metrics (≥4 valid frames); secondary: all successfully localized frames.
CASIA's “train” folder is used only for evaluation, never training or threshold fitting.
No spectral inference or auxiliary-head inference.

This is the **Kaggle extracted-image test**, not a verified official subject-disjoint CASIA benchmark. Audit train/test copy overlap, coverage and localization failures. The full-scale source threshold was calibrated using CelebA Test-as-Val, as recorded in experiment 18.


In [9]:
# CONFIG — edit only mounts / run selection as needed.
from pathlib import Path
INPUT_ROOT = Path('/kaggle/input')
CASIA_ROOT = "/kaggle/input/datasets/minhnh2107/casiafasd"  # e.g. '/kaggle/input/datasets/minhnh2107/casiafasd'
RUNS_100K_ROOT = "/kaggle/input/datasets/locngodev/models-100k/runs"  # experiment17 root OR its runs/ directory OR upload root
RUNS_FULL_ROOT = "/kaggle/input/datasets/locngodev/full-scale-models/runs"  # experiment18 root OR its runs/ directory OR upload root
SCRFD_MODEL_PATH = "/kaggle/input/datasets/locngodev/scrfd-model/det_500m.onnx"  # exact det_500m.onnx if multiple copies are mounted
OUTPUT_ROOT = Path('/kaggle/working/casia_train_test_C_P3SF_R7SC_100k_fullscale')
RUN_FILTER = None
# RUN_FILTER = ['C_100K_crop15', 'P3_SF_100K_crop15', 'R7_SC_100K_crop15']
# RUN_FILTER = ['C_FULL_crop15', 'P3_SF_FULL_crop15', 'R7_SC_FULL_crop15']
# RUN_FILTER = ['R7_SC_100K_crop15']
BATCH_SIZE = 256
LOG_EVERY_BATCHES = 5
DETECTION_LOG_EVERY = 100
MIN_VALID_FRAMES = 4
SEED = 100
INSTALL_DETECTOR_DEPS = True
# CUDA12-compatible version used successfully by notebook00d; avoid unpinned CUDA13 wheels.
ONNXRUNTIME_GPU_SPEC = 'onnxruntime-gpu==1.26.0'
CROP_SCALE = 1.5
INPUT_SIZE = 80
DETECTOR_SIZES = [(640,640), (480,480), (320,320)]
DETECTOR_CONFIDENCE = 0.5
RUN_IDS = ['C_100K_crop15', 'P3_SF_100K_crop15', 'R7_SC_100K_crop15',
           'C_FULL_crop15', 'P3_SF_FULL_crop15', 'R7_SC_FULL_crop15']
LABELS = {'C_100K_crop15':'Clean (100K)', 'P3_SF_100K_crop15':'WBST (100K)',
          'R7_SC_100K_crop15':'MAST-PAD (100K)', 'C_FULL_crop15':'Clean (full-scale)',
          'P3_SF_FULL_crop15':'WBST (full-scale)', 'R7_SC_FULL_crop15':'MAST-PAD (full-scale)'}
if RUN_FILTER is not None:
    if not isinstance(RUN_FILTER,(list,tuple)) or not RUN_FILTER or len(set(RUN_FILTER))!=len(RUN_FILTER) or set(RUN_FILTER)-set(RUN_IDS):
        raise ValueError('RUN_FILTER must be None or a nonempty list of unique RUN_IDS')
SELECTED_RUNS = RUN_IDS if RUN_FILTER is None else list(RUN_FILTER)
if BATCH_SIZE<1 or MIN_VALID_FRAMES<1: raise ValueError('Invalid batch / minimum-frame setting')
print('Selected runs:', SELECTED_RUNS, flush=True)


Selected runs: ['C_100K_crop15', 'P3_SF_100K_crop15', 'R7_SC_100K_crop15', 'C_FULL_crop15', 'P3_SF_FULL_crop15', 'R7_SC_FULL_crop15']


In [10]:
# SETUP — import PyTorch first to expose its CUDA/cuDNN libraries.
import os,sys,subprocess,random,time,gc,importlib.util
from importlib.metadata import version,PackageNotFoundError
import torch
print('PyTorch:',torch.__version__, 'CUDA:',torch.version.cuda, 'GPU:',torch.cuda.is_available(),flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Enable Kaggle Settings > Accelerator > GPU and restart the session.')
def installed(name):
    try:return version(name)
    except PackageNotFoundError:return None
if INSTALL_DETECTOR_DEPS:
    ready=(installed('insightface')=='2.0' and installed('onnxruntime-gpu')=='1.26.0' and installed('onnxruntime') is None)
    if not ready:
        if 'onnxruntime' in sys.modules or 'insightface' in sys.modules:
            raise RuntimeError('Restart the Kaggle session before changing detector dependencies.')
        subprocess.check_call([sys.executable,'-m','pip','install','-q','insightface==2.0'])
        subprocess.check_call([sys.executable,'-m','pip','uninstall','-y','onnxruntime','onnxruntime-gpu'])
        subprocess.check_call([sys.executable,'-m','pip','install','-q',ONNXRUNTIME_GPU_SPEC])
import cv2,numpy as np,pandas as pd,json,re,math,zipfile,types
from torch import nn
from sklearn.metrics import roc_curve,roc_auc_score
from IPython.display import display,FileLink
import matplotlib.pyplot as plt
import onnxruntime as ort
from insightface import model_zoo
if hasattr(ort,'preload_dlls'):ort.preload_dlls()
print('ONNX Runtime:',ort.__version__,'providers:',ort.get_available_providers(),flush=True)
if 'CUDAExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('CUDAExecutionProvider unavailable; restart and rerun the setup cell.')
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
cv2.setNumThreads(1)
device=torch.device('cuda')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)


PyTorch: 2.11.0+cu128 CUDA: 12.8 GPU: True
ONNX Runtime: 1.26.0 providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


## Embedded inference architecture

MiniFASNet source copied from `antispoof/models/MiniFASNet.py`; binary wrapper, crop and metric formulas copied from notebook18 (shared with notebook17). Auxiliary modules exist only to strictly load the training checkpoint; the inference forward emits two PAD logits.


In [11]:
# -*- coding: utf-8 -*-
# @Time : 20-6-3 下午4:45
# @Author : zhuying
# @Company : Minivision
# @File : MiniFASNet.py
# @Software : PyCharm
import torch
import torch.nn.functional as F
from torch.nn import Linear, Conv2d, BatchNorm1d, BatchNorm2d, PReLU, ReLU, Sigmoid, \
    AdaptiveAvgPool2d, Sequential, Module


class L2Norm(Module):
    def forward(self, input):
        return F.normalize(input)


class Flatten(Module):
    def forward(self, input):
        return input.view(input.size(0), -1)


class Conv_block(Module):
    def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
        super(Conv_block, self).__init__()
        self.conv = Conv2d(in_c, out_c, kernel_size=kernel, groups=groups,
                           stride=stride, padding=padding, bias=False)
        self.bn = BatchNorm2d(out_c)
        self.prelu = PReLU(out_c)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.prelu(x)
        return x


class Linear_block(Module):
    def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
        super(Linear_block, self).__init__()
        self.conv = Conv2d(in_c, out_channels=out_c, kernel_size=kernel,
                           groups=groups, stride=stride, padding=padding, bias=False)
        self.bn = BatchNorm2d(out_c)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return x


class Depth_Wise(Module):
     def __init__(self, c1, c2, c3, residual=False, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=1):
        super(Depth_Wise, self).__init__()
        c1_in, c1_out = c1
        c2_in, c2_out = c2
        c3_in, c3_out = c3
        self.conv = Conv_block(c1_in, out_c=c1_out, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.conv_dw = Conv_block(c2_in, c2_out, groups=c2_in, kernel=kernel, padding=padding, stride=stride)
        self.project = Linear_block(c3_in, c3_out, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.residual = residual

     def forward(self, x):
        if self.residual:
            short_cut = x
        x = self.conv(x)
        x = self.conv_dw(x)
        x = self.project(x)
        if self.residual:
            output = short_cut + x
        else:
            output = x
        return output


class Residual(Module):
    def __init__(self, c1, c2, c3, num_block, groups, kernel=(3, 3), stride=(1, 1), padding=(1, 1)):
        super(Residual, self).__init__()
        modules = []
        for i in range(num_block):
            c1_tuple = c1[i]
            c2_tuple = c2[i]
            c3_tuple = c3[i]
            modules.append(Depth_Wise(c1_tuple, c2_tuple, c3_tuple, residual=True,
                                      kernel=kernel, padding=padding, stride=stride, groups=groups))
        self.model = Sequential(*modules)

    def forward(self, x):
        return self.model(x)


class SEModule(Module):
    def __init__(self, channels, reduction):
        super(SEModule, self).__init__()
        self.avg_pool = AdaptiveAvgPool2d(1)
        self.fc1 = Conv2d(
            channels, channels // reduction, kernel_size=1, padding=0, bias=False)
        self.bn1 = BatchNorm2d(channels // reduction)
        self.relu = ReLU(inplace=True)
        self.fc2 = Conv2d(
            channels // reduction, channels, kernel_size=1, padding=0, bias=False)
        self.bn2 = BatchNorm2d(channels)
        self.sigmoid = Sigmoid()

    def forward(self, x):
        module_input = x
        x = self.avg_pool(x)
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.sigmoid(x)
        return module_input * x


class ResidualSE(Module):
    def __init__(self, c1, c2, c3, num_block, groups, kernel=(3, 3), stride=(1, 1), padding=(1, 1), se_reduct=4):
        super(ResidualSE, self).__init__()
        modules = []
        for i in range(num_block):
            c1_tuple = c1[i]
            c2_tuple = c2[i]
            c3_tuple = c3[i]
            if i == num_block-1:
                modules.append(
                    Depth_Wise_SE(c1_tuple, c2_tuple, c3_tuple, residual=True, kernel=kernel, padding=padding, stride=stride,
                               groups=groups, se_reduct=se_reduct))
            else:
                modules.append(Depth_Wise(c1_tuple, c2_tuple, c3_tuple, residual=True, kernel=kernel, padding=padding,
                                          stride=stride, groups=groups))
        self.model = Sequential(*modules)

    def forward(self, x):
        return self.model(x)


class Depth_Wise_SE(Module):
    def __init__(self, c1, c2, c3, residual=False, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=1, se_reduct=8):
        super(Depth_Wise_SE, self).__init__()
        c1_in, c1_out = c1
        c2_in, c2_out = c2
        c3_in, c3_out = c3
        self.conv = Conv_block(c1_in, out_c=c1_out, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.conv_dw = Conv_block(c2_in, c2_out, groups=c2_in, kernel=kernel, padding=padding, stride=stride)
        self.project = Linear_block(c3_in, c3_out, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.residual = residual
        self.se_module = SEModule(c3_out, se_reduct)

    def forward(self, x):
        if self.residual:
            short_cut = x
        x = self.conv(x)
        x = self.conv_dw(x)
        x = self.project(x)
        if self.residual:
            x = self.se_module(x)
            output = short_cut + x
        else:
            output = x
        return output


class MiniFASNet(Module):
    def __init__(self, keep, embedding_size, conv6_kernel=(7, 7),
                 drop_p=0.0, num_classes=3, img_channel=3):
        super(MiniFASNet, self).__init__()
        self.embedding_size = embedding_size

        self.conv1 = Conv_block(img_channel, keep[0], kernel=(3, 3), stride=(2, 2), padding=(1, 1))
        self.conv2_dw = Conv_block(keep[0], keep[1], kernel=(3, 3), stride=(1, 1), padding=(1, 1), groups=keep[1])

        c1 = [(keep[1], keep[2])]
        c2 = [(keep[2], keep[3])]
        c3 = [(keep[3], keep[4])]

        self.conv_23 = Depth_Wise(c1[0], c2[0], c3[0], kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=keep[3])

        c1 = [(keep[4], keep[5]), (keep[7], keep[8]), (keep[10], keep[11]), (keep[13], keep[14])]
        c2 = [(keep[5], keep[6]), (keep[8], keep[9]), (keep[11], keep[12]), (keep[14], keep[15])]
        c3 = [(keep[6], keep[7]), (keep[9], keep[10]), (keep[12], keep[13]), (keep[15], keep[16])]

        self.conv_3 = Residual(c1, c2, c3, num_block=4, groups=keep[4], kernel=(3, 3), stride=(1, 1), padding=(1, 1))

        c1 = [(keep[16], keep[17])]
        c2 = [(keep[17], keep[18])]
        c3 = [(keep[18], keep[19])]

        self.conv_34 = Depth_Wise(c1[0], c2[0], c3[0], kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=keep[19])

        c1 = [(keep[19], keep[20]), (keep[22], keep[23]), (keep[25], keep[26]), (keep[28], keep[29]),
              (keep[31], keep[32]), (keep[34], keep[35])]
        c2 = [(keep[20], keep[21]), (keep[23], keep[24]), (keep[26], keep[27]), (keep[29], keep[30]),
              (keep[32], keep[33]), (keep[35], keep[36])]
        c3 = [(keep[21], keep[22]), (keep[24], keep[25]), (keep[27], keep[28]), (keep[30], keep[31]),
              (keep[33], keep[34]), (keep[36], keep[37])]

        self.conv_4 = Residual(c1, c2, c3, num_block=6, groups=keep[19], kernel=(3, 3), stride=(1, 1), padding=(1, 1))

        c1 = [(keep[37], keep[38])]
        c2 = [(keep[38], keep[39])]
        c3 = [(keep[39], keep[40])]

        self.conv_45 = Depth_Wise(c1[0], c2[0], c3[0], kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=keep[40])

        c1 = [(keep[40], keep[41]), (keep[43], keep[44])]
        c2 = [(keep[41], keep[42]), (keep[44], keep[45])]
        c3 = [(keep[42], keep[43]), (keep[45], keep[46])]

        self.conv_5 = Residual(c1, c2, c3, num_block=2, groups=keep[40], kernel=(3, 3), stride=(1, 1), padding=(1, 1))
        self.conv_6_sep = Conv_block(keep[46], keep[47], kernel=(1, 1), stride=(1, 1), padding=(0, 0))
        self.conv_6_dw = Linear_block(keep[47], keep[48], groups=keep[48], kernel=conv6_kernel, stride=(1, 1), padding=(0, 0))
        self.conv_6_flatten = Flatten()
        self.linear = Linear(512, embedding_size, bias=False)
        self.bn = BatchNorm1d(embedding_size)
        self.drop = torch.nn.Dropout(p=drop_p)
        self.prob = Linear(embedding_size, num_classes, bias=False)

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2_dw(out)
        out = self.conv_23(out)
        out = self.conv_3(out)
        out = self.conv_34(out)
        out = self.conv_4(out)
        out = self.conv_45(out)
        out = self.conv_5(out)
        out = self.conv_6_sep(out)
        out = self.conv_6_dw(out)
        out = self.conv_6_flatten(out)
        if self.embedding_size != 512:
            out = self.linear(out)
        out = self.bn(out)
        out = self.drop(out)
        out = self.prob(out)
        return out


class MiniFASNetSE(MiniFASNet):
    def __init__(self, keep, embedding_size, conv6_kernel=(7, 7),drop_p=0.75, num_classes=4, img_channel=3):
        super(MiniFASNetSE, self).__init__(keep=keep, embedding_size=embedding_size, conv6_kernel=conv6_kernel,
                                               drop_p=drop_p, num_classes=num_classes, img_channel=img_channel)

        c1 = [(keep[4], keep[5]), (keep[7], keep[8]), (keep[10], keep[11]), (keep[13], keep[14])]
        c2 = [(keep[5], keep[6]), (keep[8], keep[9]), (keep[11], keep[12]), (keep[14], keep[15])]
        c3 = [(keep[6], keep[7]), (keep[9], keep[10]), (keep[12], keep[13]), (keep[15], keep[16])]

        self.conv_3 = ResidualSE(c1, c2, c3, num_block=4, groups=keep[4], kernel=(3, 3), stride=(1, 1), padding=(1, 1))

        c1 = [(keep[19], keep[20]), (keep[22], keep[23]), (keep[25], keep[26]), (keep[28], keep[29]),
              (keep[31], keep[32]), (keep[34], keep[35])]
        c2 = [(keep[20], keep[21]), (keep[23], keep[24]), (keep[26], keep[27]), (keep[29], keep[30]),
              (keep[32], keep[33]), (keep[35], keep[36])]
        c3 = [(keep[21], keep[22]), (keep[24], keep[25]), (keep[27], keep[28]), (keep[30], keep[31]),
              (keep[33], keep[34]), (keep[36], keep[37])]

        self.conv_4 = ResidualSE(c1, c2, c3, num_block=6, groups=keep[19], kernel=(3, 3), stride=(1, 1), padding=(1, 1))

        c1 = [(keep[40], keep[41]), (keep[43], keep[44])]
        c2 = [(keep[41], keep[42]), (keep[44], keep[45])]
        c3 = [(keep[42], keep[43]), (keep[45], keep[46])]
        self.conv_5 = ResidualSE(c1, c2, c3, num_block=2, groups=keep[40], kernel=(3, 3), stride=(1, 1), padding=(1, 1))



keep_dict = {'1.8M': [32, 32, 103, 103, 64, 13, 13, 64, 26, 26,
                      64, 13, 13, 64, 52, 52, 64, 231, 231, 128,
                      154, 154, 128, 52, 52, 128, 26, 26, 128, 52,
                      52, 128, 26, 26, 128, 26, 26, 128, 308, 308,
                      128, 26, 26, 128, 26, 26, 128, 512, 512],

             '1.8M_': [32, 32, 103, 103, 64, 13, 13, 64, 13, 13, 64, 13,
                       13, 64, 13, 13, 64, 231, 231, 128, 231, 231, 128, 52,
                       52, 128, 26, 26, 128, 77, 77, 128, 26, 26, 128, 26, 26,
                       128, 308, 308, 128, 26, 26, 128, 26, 26, 128, 512, 512]
             }


# (80x80) flops: 0.044, params: 0.41
def MiniFASNetV1(embedding_size=128, conv6_kernel=(7, 7),
                     drop_p=0.2, num_classes=3, img_channel=3):
    return MiniFASNet(keep_dict['1.8M'], embedding_size, conv6_kernel, drop_p, num_classes, img_channel)


# (80x80) flops: 0.044, params: 0.43
def MiniFASNetV2(embedding_size=128, conv6_kernel=(7, 7),
                     drop_p=0.2, num_classes=3, img_channel=3):
    return MiniFASNet(keep_dict['1.8M_'], embedding_size, conv6_kernel, drop_p, num_classes, img_channel)

def MiniFASNetV1SE(embedding_size=128, conv6_kernel=(7, 7),
                   drop_p=0.75, num_classes=3, img_channel=3):
    return MiniFASNetSE(keep_dict['1.8M'], embedding_size, conv6_kernel,drop_p, num_classes, img_channel)

# (80x80) flops: 0.044, params: 0.43
def MiniFASNetV2SE(embedding_size=128, conv6_kernel=(7, 7),
                   drop_p=0.75, num_classes=4, img_channel=3):
    return MiniFASNetSE(keep_dict['1.8M_'], embedding_size, conv6_kernel,drop_p, num_classes, img_channel)

module=types.SimpleNamespace(MiniFASNetV2=MiniFASNetV2)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return cv2.resize(patch,(size,size))

def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}


In [12]:
# Mount discovery and helpers. No hashing of whole resource trees or contract checks.
def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):
    Path(path).parent.mkdir(parents=True,exist_ok=True)
    Path(path).write_text(json.dumps(value,indent=2,allow_nan=False)+'\n')
def one_file(explicit,name):
    paths=[Path(explicit)] if explicit is not None else sorted(INPUT_ROOT.rglob(name))
    paths=sorted({p.resolve() for p in paths if p.is_file()},key=str)
    if len(paths)!=1:raise RuntimeError(f'Expected one {name}, found {paths}; set the exact path in config.')
    return paths[0]
def resolve_casia():
    if CASIA_ROOT is not None:
        p=Path(CASIA_ROOT)
        if not p.is_dir():raise FileNotFoundError(p)
        return p
    folders=sorted({p.parent.parent.parent for p in INPUT_ROOT.rglob('color')
                    if p.is_dir() and p.parent.name=='test_img' and p.parent.parent.name=='test_img'},key=str)
    if len(folders)!=1:raise RuntimeError(f'Expected one CASIA image-copy root, found {folders}; set CASIA_ROOT.')
    return folders[0]
def resolve_run(run_id):
    explicit=RUNS_100K_ROOT if '100K' in run_id else RUNS_FULL_ROOT
    root=INPUT_ROOT if explicit is None else Path(explicit)
    if not root.is_dir():raise FileNotFoundError(root)
    candidates=[]
    if root.name==run_id and (root/'config.json').is_file():candidates.append(root)
    candidates += [p for p in root.rglob(run_id) if p.is_dir() and (p/'config.json').is_file()]
    candidates=sorted(set(candidates),key=str)
    if len(candidates)!=1:raise RuntimeError(f'{run_id}: found {candidates}; set the appropriate RUNS_*_ROOT.')
    return candidates[0]
def load_run(run_id):
    folder=resolve_run(run_id);cfg=read_json(folder/'config.json')
    if cfg['run_id']!=run_id:raise ValueError('Run config ID mismatch: '+run_id)
    if cfg.get('input')!='80x80 BGR float [0,1] crop1.5':raise ValueError('Unexpected input preprocessing: '+run_id)
    if any(cfg[k]!=CROP_SCALE for k in ['train_crop_factor','val_crop_factor','test_crop_factor','lcc_crop_factor']):
        raise ValueError('Expected crop1.5 for every selected run')
    checkpoint_path=folder/'best.pth'
    checkpoint=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
    if checkpoint['run_id']!=run_id or checkpoint['config']!=cfg:raise ValueError('Best checkpoint/config mismatch: '+run_id)
    frozen_path=folder/'evaluation_freeze.json'
    if not frozen_path.is_file():frozen_path=folder/'completion.json'
    frozen=read_json(frozen_path)
    threshold=float(frozen['locked_threshold'])
    if frozen['run_id']!=run_id or frozen['best_epoch']!=checkpoint['epoch'] or not math.isfinite(threshold):
        raise ValueError('Frozen best epoch/threshold mismatch: '+run_id)
    model=PADNet(True).eval();model.load_state_dict(checkpoint['state_dict'],strict=True)
    source='full official CelebA Test-as-Val' if 'FULL' in run_id else 'CelebA Val15K'
    info={'run_id':run_id,'method':LABELS[run_id],'scale':'full-scale' if 'FULL' in run_id else '100K',
          'checkpoint_path':str(checkpoint_path),'config_path':str(folder/'config.json'),
          'frozen_threshold_path':str(frozen_path),'best_epoch':int(checkpoint['epoch']),
          'threshold':threshold,'threshold_source':source,'amp':bool(cfg.get('amp',True)),
          'train_seed':cfg.get('train_seed'),'input':cfg['input']}
    del checkpoint
    return model,info
CASIA=resolve_casia();detector_path=one_file(SCRFD_MODEL_PATH,'det_500m.onnx')
print('CASIA:',CASIA,'\nDetector:',detector_path,flush=True)
# Cheap checkpoint preflight before localization; release each model after checking.
provenance=[]
for run_id in SELECTED_RUNS:
    model,info=load_run(run_id);provenance.append(info);del model;gc.collect()
    print(f'[Ready] {run_id}: best epoch {info["best_epoch"]}, frozen threshold {info["threshold"]}',flush=True)
write_json(OUTPUT_ROOT/'model_provenance.json',provenance)


CASIA: /kaggle/input/datasets/minhnh2107/casiafasd 
Detector: /kaggle/input/datasets/locngodev/scrfd-model/det_500m.onnx
[Ready] C_100K_crop15: best epoch 15, frozen threshold 0.1822357177734375
[Ready] P3_SF_100K_crop15: best epoch 15, frozen threshold 0.29180908203125
[Ready] R7_SC_100K_crop15: best epoch 11, frozen threshold -0.373016357421875
[Ready] C_FULL_crop15: best epoch 9, frozen threshold 5.130859375
[Ready] P3_SF_FULL_crop15: best epoch 6, frozen threshold 5.3037109375
[Ready] R7_SC_FULL_crop15: best epoch 7, frozen threshold 4.66259765625


In [13]:
# Full train/test-color manifests, validated explicit filename labels; no frame subsampling.
EXPECTED_CODES={'1':(0,'real','low'),'2':(0,'real','normal'),'HR_1':(0,'real','high'),
 '3':(1,'warped','low'),'4':(1,'warped','normal'),'HR_2':(1,'warped','high'),
 '5':(1,'cut','low'),'6':(1,'cut','normal'),'HR_3':(1,'cut','high'),
 '7':(1,'replay','low'),'8':(1,'replay','normal'),'HR_4':(1,'replay','high')}
FILENAME_RE=re.compile(r'^(?P<subject>\d+)_(?P<code>HR_[1-4]|[1-8])\.avi_(?P<frame>\d+)_(?P<label>real|fake)\.jpg$',re.I)
def read_split(folder):
    output=[]
    for path in sorted(folder.iterdir()):
        if path.name.startswith('.'):continue
        match=FILENAME_RE.fullmatch(path.name)
        if not path.is_file() or not match:raise ValueError('Unexpected CASIA filename: '+str(path))
        g=match.groupdict();code=g['code'].upper();label,attack,quality=EXPECTED_CODES[code]
        if g['label'].lower()!=('real' if label==0 else 'fake'):raise ValueError('Filename label/code conflict: '+str(path))
        output.append({'subject_id':int(g['subject']),'video_code':code,'video_id':f'{int(g["subject"])}_{code}.avi',
                       'frame_index':int(g['frame']),'image_path':str(path),'binary_label':label,
                       'attack_type':attack,'quality':quality})
    frame=pd.DataFrame(output)
    if frame.empty or frame.duplicated(['video_id','frame_index']).any():raise ValueError('Empty/duplicate manifest: '+str(folder))
    return frame.sort_values(['subject_id','video_id','frame_index']).reset_index(drop=True)
# ID collision checks use explicit labels, never PAD scores or detection results.
train_names=read_split(CASIA/'train_img/train_img/color').assign(split='train')
test_names=read_split(CASIA/'test_img/test_img/color').assign(split='test')
frame_manifest=pd.concat([train_names,test_names],ignore_index=True)
for vid,group in frame_manifest.groupby('video_id'):
    if len(group[['subject_id','video_code','binary_label','attack_type','quality']].drop_duplicates())!=1:
        raise ValueError('Inconsistent video labels across splits: '+vid)
def combined_unique(frame):
    # Stable train then test ordering; test takes precedence independently of detection success.
    ordered=pd.concat([frame.loc[frame.split=='train'],frame.loc[frame.split=='test']],ignore_index=True)
    result=ordered.drop_duplicates(['video_id','frame_index'],keep='last').copy()
    result['source_split']=result['split']
    result['split']='combined'
    return result.sort_values(['subject_id','video_id','frame_index']).reset_index(drop=True)
combined_manifest=combined_unique(frame_manifest)
POOLS=['train','test','combined']
pool_manifests={'train':train_names,'test':test_names,'combined':combined_manifest}
def video_manifest(frame):
    return frame.groupby(['split','video_id'],sort=True).agg(subject_id=('subject_id','first'),
      video_code=('video_code','first'),binary_label=('binary_label','first'),
      attack_type=('attack_type','first'),quality=('quality','first'),candidate_frames=('frame_index','size')).reset_index()
videos=pd.concat([video_manifest(pool_manifests[split]) for split in POOLS],ignore_index=True)
for split,frame in pool_manifests.items():
    if set(frame.binary_label)!={0,1}:raise ValueError('Real and Attack required in '+split)
    frame.to_csv(OUTPUT_ROOT/f'casia_{split}_frame_manifest.csv',index=False)
frame_manifest.to_csv(OUTPUT_ROOT/'casia_frame_manifest.csv',index=False)
videos.to_csv(OUTPUT_ROOT/'casia_video_manifest.csv',index=False)
overlap={'subjects':sorted(set(test_names.subject_id)&set(train_names.subject_id)),
         'video_ids':len(set(test_names.video_id)&set(train_names.video_id)),
         'frame_ids':len(set(zip(test_names.video_id,test_names.frame_index))&set(zip(train_names.video_id,train_names.frame_index)))}
populations={split:{'candidate_frames':len(pool_manifests[split]),
                    'candidate_videos':int((videos.split==split).sum())} for split in POOLS}
protocol={'dataset':'CASIA-FASD Kaggle extracted-image train + test evaluation (minhnh2107/casiafasd)',
 'official_subject_disjoint_protocol_verified':False,'splits':['train_img/train_img/color','test_img/test_img/color'],
 'train_test_copy_overlap':overlap,'frame_sampling':'none; all available train and test color JPEGs',
 'raw_train_test_frames':len(frame_manifest),'populations':populations,
 'combined_policy':'unique (video_id,frame_index); prefer test copy before detection; no duplicate weighting','selected_runs':SELECTED_RUNS,
 'detector':'SCRFD-500M','detector_path':str(detector_path),'detector_sizes':DETECTOR_SIZES,
 'detector_confidence':DETECTOR_CONFIDENCE,'face_selection':'largest positive finite bbox; confidence tie-break',
 'crop_factor':CROP_SCALE,'input_size':INPUT_SIZE,'color_order':'BGR','input_range':'[0,1]',
 'score':'logit(real)-logit(attack)','primary_unit':'reconstructed video; mean valid-frame score',
 'min_valid_frames':MIN_VALID_FRAMES,'secondary_unit':'all successfully localized frames in the specified pool',
 'threshold_policy':'each run frozen source threshold; never fit on CASIA',
 'eer_policy':'descriptive ROC EER only; its target threshold is never used for predictions',
 'target_training':False,'target_adaptation':False,'target_threshold_calibration':False,
 'full_scale_source_selection':'CelebA Test-as-Val, as in experiment18',
 'inference_precision':'per-run original AMP setting; logits converted to float32 before subtraction',
 'seed':SEED,'batch_size':BATCH_SIZE,'torch_version':torch.__version__,'ort_version':ort.__version__}
write_json(OUTPUT_ROOT/'evaluation_protocol.json',protocol)
print('Candidate populations:',populations,flush=True)
print('Train/test image-copy overlap:',overlap,flush=True)
display(videos.groupby(['split','binary_label','attack_type','quality']).size().rename('videos').reset_index())


Candidate populations: {'train': {'candidate_frames': 1655, 'candidate_videos': 240}, 'test': {'candidate_frames': 2408, 'candidate_videos': 360}, 'combined': {'candidate_frames': 2660, 'candidate_videos': 360}}
Train/test image-copy overlap: {'subjects': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20], 'video_ids': 240, 'frame_ids': 1403}


,split,binary_label,attack_type,quality,videos
0,combined,0,real,high,30
1,combined,0,real,low,30
2,combined,0,real,normal,30
3,combined,1,cut,high,30
4,combined,1,cut,low,30
5,combined,1,cut,normal,30
6,combined,1,replay,high,30
7,combined,1,replay,low,30
8,combined,1,replay,normal,30
9,combined,1,warped,high,30


In [14]:
# Localize once and cache crops in RAM. Every selected model receives identical tensors.
# Repeat this cell only to rebuild detection; no target bbox/GT fallback is used.
detector=model_zoo.get_model(str(detector_path),providers=['CUDAExecutionProvider','CPUExecutionProvider'])
if detector is None:raise RuntimeError('Could not load SCRFD-500M')
detector.prepare(ctx_id=0,input_size=DETECTOR_SIZES[0],det_thresh=DETECTOR_CONFIDENCE)
active=detector.session.get_providers()
# CPU provider may handle shape operations; CUDA must actually be active.
if 'CUDAExecutionProvider' not in active:raise RuntimeError('SCRFD CUDA failed to load; active providers: '+str(active))
print('SCRFD active providers:',active,flush=True)
def detect_boxes(image,size):
    try:return detector.detect(image,input_size=size,max_num=0,metric='default')[0]
    except TypeError:
        previous=detector.input_size;detector.input_size=size
        try:return detector.detect(image,max_num=0,metric='default')[0]
        finally:detector.input_size=previous
patches=[];records=[];started=time.monotonic()
print('[Detection] Starting shared full train/test localization',flush=True)
for i,row in enumerate(frame_manifest.to_dict('records')):
    rec=dict(row,decoded=False,detected=False,crop_index=-1,bbox_xyxy='',confidence=None,detector_size=None,failure_reason='')
    image=cv2.imread(row['image_path'],cv2.IMREAD_COLOR)
    if image is None:rec['failure_reason']='decode_failed'
    else:
        rec['decoded']=True
        try:
            for size in DETECTOR_SIZES:
                boxes=detect_boxes(image,size)
                if boxes is None:continue
                boxes=[b for b in boxes if len(b)>=5 and np.isfinite(b[:5]).all() and b[2]>b[0] and b[3]>b[1] and b[4]>=DETECTOR_CONFIDENCE]
                if not boxes:continue
                box=max(boxes,key=lambda b:((float(b[2])-float(b[0]))*(float(b[3])-float(b[1])),float(b[4])))
                patch=mini_crop_bgr(image,box[:4])
                rec.update(detected=True,crop_index=len(patches),bbox_xyxy=json.dumps([float(v) for v in box[:4]]),
                           confidence=float(box[4]),detector_size=size[0])
                patches.append(patch);break
            if not rec['detected']:rec['failure_reason']='scrfd_no_face'
        except Exception as exc:rec['failure_reason']=f'{type(exc).__name__}: {exc}'
    records.append(rec)
    if i==0 or (i+1)%DETECTION_LOG_EVERY==0 or i+1==len(frame_manifest):
        print(f'[Detection] {i+1}/{len(frame_manifest)} frames; valid={len(patches)}; elapsed={time.monotonic()-started:.1f}s',flush=True)
    if (i+1)%DETECTION_LOG_EVERY==0:pd.DataFrame(records).to_csv(OUTPUT_ROOT/'casia_frame_detection.partial.csv',index=False)
detection=pd.DataFrame(records);detection.to_csv(OUTPUT_ROOT/'casia_frame_detection.csv',index=False)
if not patches:raise RuntimeError('No valid face crops; inspect casia_frame_detection.csv')
CROPS=np.stack(patches);del patches
valid=detection.loc[detection.detected].copy().reset_index(drop=True)
assert valid.crop_index.tolist()==list(range(len(CROPS)))
np.save(OUTPUT_ROOT/'shared_crops_bgr_uint8.npy',CROPS)
# Pooled rows are fixed by the pre-detection manifest policy, including failed test copies.
pool_detection={'train':detection.loc[detection.split=='train'].copy(),
                'test':detection.loc[detection.split=='test'].copy(),
                'combined':combined_unique(detection)}
combined_rows=pool_detection['combined']
assert combined_rows.image_path.tolist()==combined_manifest.image_path.tolist()
combined_rows.to_csv(OUTPUT_ROOT/'casia_combined_frame_detection.csv',index=False)
coverages=[]
for split in POOLS:
    frame=pool_detection[split]
    cov=frame.groupby(['split','video_id'],sort=True).agg(decoded_frames=('decoded','sum'),valid_frames=('detected','sum')).reset_index()
    cov=videos.loc[videos.split==split].merge(cov,on=['split','video_id'],validate='one_to_one')
    cov['scorable']=cov.valid_frames>=MIN_VALID_FRAMES
    cov['detection_coverage']=cov.valid_frames/cov.candidate_frames
    coverages.append(cov)
coverage=pd.concat(coverages,ignore_index=True)
coverage.to_csv(OUTPUT_ROOT/'casia_video_coverage.csv',index=False)
valid.groupby(['split','binary_label','attack_type','quality']).size().rename('valid_frames').reset_index().to_csv(OUTPUT_ROOT/'valid_frames_by_group.csv',index=False)
protocol['detector_active_providers']=active
write_json(OUTPUT_ROOT/'evaluation_protocol.json',protocol)
print(f'[Detection complete] {len(valid)}/{len(frame_manifest)} raw train/test frames',flush=True)
display(coverage.groupby('split').agg(candidate_videos=('video_id','size'),scorable_videos=('scorable','sum'),candidate_frames=('candidate_frames','sum'),valid_frames=('valid_frames','sum')).reset_index())
display(detection.failure_reason.value_counts())
del detector;gc.collect();torch.cuda.empty_cache()


SCRFD active providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
[Detection] Starting shared full train/test localization
[Detection] 1/4063 frames; valid=1; elapsed=5.0s
[Detection] 100/4063 frames; valid=100; elapsed=7.0s
[Detection] 200/4063 frames; valid=200; elapsed=8.8s
[Detection] 300/4063 frames; valid=300; elapsed=12.8s
[Detection] 400/4063 frames; valid=400; elapsed=14.6s
[Detection] 500/4063 frames; valid=500; elapsed=16.5s
[Detection] 600/4063 frames; valid=600; elapsed=18.5s
[Detection] 700/4063 frames; valid=700; elapsed=20.4s
[Detection] 800/4063 frames; valid=800; elapsed=22.4s
[Detection] 900/4063 frames; valid=900; elapsed=24.4s
[Detection] 1000/4063 frames; valid=1000; elapsed=26.4s
[Detection] 1100/4063 frames; valid=1100; elapsed=28.4s
[Detection] 1200/4063 frames; valid=1200; elapsed=30.3s
[Detection] 1300/4063 frames; valid=1300; elapsed=32.2s
[Detection] 1400/4063 frames; valid=1400; elapsed=34.2s
[Detection] 1500/4063 frames; valid=1500; elapsed=36.1s


,split,candidate_videos,scorable_videos,candidate_frames,valid_frames
0,combined,360,358,2660,2660
1,test,360,352,2408,2408
2,train,240,237,1655,1655


failure_reason
    4063
Name: count, dtype: int64

In [15]:
# Frozen checkpoint inference, one model at a time. Progress printed immediately and per batch.
def safe_metrics(labels,scores,threshold,candidate_n):
    if len(labels)==0 or set(np.asarray(labels)==0)!={False,True}:
        y=np.asarray(labels);pred=np.asarray(scores)>=threshold;real=y==0;attack=y!=0
        return dict(auc=None,eer=None,tpr_at_fpr_1pct=None,
                    apcer=float(pred[attack].mean()) if attack.any() else None,
                    bpcer=float((~pred[real]).mean()) if real.any() else None,acer=None,hter=None,
                    binary_accuracy=float((pred==real).mean()) if len(y) else None,
                    locked_source_threshold=float(threshold),candidate_n=int(candidate_n),
                    scored_n=len(labels),detector_coverage=len(labels)/candidate_n if candidate_n else 0)
    return metrics(labels,scores,threshold,candidate_n)
summary=[];all_frame_predictions=[];all_video_predictions=[];subgroup_rows=[]
for run_id in SELECTED_RUNS:
    model,info=load_run(run_id);model=model.to(device);threshold=info['threshold']
    run_output=OUTPUT_ROOT/'runs'/run_id;run_output.mkdir(parents=True,exist_ok=True)
    started=time.monotonic();chunks=[];num_batches=math.ceil(len(CROPS)/BATCH_SIZE)
    print(f'\n[Inference] {run_id}: {len(CROPS)} shared crops, {num_batches} batches; threshold={threshold}',flush=True)
    with torch.inference_mode():
        for batch_no,start in enumerate(range(0,len(CROPS),BATCH_SIZE),1):
            arr=np.ascontiguousarray(CROPS[start:start+BATCH_SIZE].transpose(0,3,1,2))
            x=torch.from_numpy(arr).to(device=device,dtype=torch.float32).div_(255.)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=info['amp']):z=model(x)
            z=z.float().cpu().numpy()
            if z.shape!=(len(arr),2) or not np.isfinite(z).all():raise ValueError('Nonfinite / nonbinary model logits: '+run_id)
            chunks.append(z)
            if batch_no==1 or batch_no%LOG_EVERY_BATCHES==0 or batch_no==num_batches:
                print(f'[Inference] {run_id} batch {batch_no}/{num_batches}; elapsed={time.monotonic()-started:.1f}s',flush=True)
    logits=np.concatenate(chunks)
    fp=valid[['split','subject_id','video_id','video_code','frame_index','image_path','binary_label','attack_type','quality']].copy()
    fp['run_id']=run_id;fp['logit_real']=logits[:,0];fp['logit_attack']=logits[:,1]
    fp['pad_score']=logits[:,0]-logits[:,1];fp['threshold']=threshold
    fp['predicted_real']=fp.pad_score>=threshold
    fp.to_csv(run_output/'frame_predictions.csv',index=False)
    all_frame_predictions.append(fp)
    write_json(run_output/'config.json',info|{'evaluation_protocol_file':'../../evaluation_protocol.json','training':False,'pools':POOLS})
    for split in POOLS:
        # Reindex predictions to candidate rows chosen BEFORE detection. No fallback to a train
        # duplicate when its preferred test copy failed; this prevents outcome-based selection.
        candidates=pool_detection[split]
        chosen=candidates.loc[candidates.detected].copy()
        columns=['image_path','logit_real','logit_attack','pad_score','threshold','predicted_real']
        sf=chosen.merge(fp[columns],on='image_path',how='left',validate='one_to_one')
        if sf.pad_score.isna().any():raise ValueError('Missing predictions in '+split)
        means=sf.groupby('video_id',sort=True).agg(video_score=('pad_score','mean')).reset_index()
        cv=coverage.loc[coverage.split==split].copy()
        vp=cv.merge(means,on='video_id',how='left',validate='one_to_one')
        vp['run_id']=run_id;vp['threshold']=threshold
        vp['predicted_real']=pd.Series(pd.NA,index=vp.index,dtype='boolean')
        vp.loc[vp.scorable,'predicted_real']=(vp.loc[vp.scorable,'video_score']>=threshold).to_numpy()
        scored=vp.loc[vp.scorable]
        video_metrics=safe_metrics(scored.binary_label.to_numpy(),scored.video_score.to_numpy(),threshold,len(cv))
        frame_metrics=safe_metrics(sf.binary_label.to_numpy(),sf.pad_score.to_numpy(),threshold,len(candidates))
        split_output=run_output/split;split_output.mkdir(exist_ok=True)
        sf.to_csv(split_output/'frame_predictions.csv',index=False);vp.to_csv(split_output/'video_predictions.csv',index=False)
        write_json(split_output/'video_metrics.json',video_metrics);write_json(split_output/'frame_metrics.json',frame_metrics)
        summary.append(info|{'split':split}|video_metrics|{'frame_'+k:v for k,v in frame_metrics.items()})
        all_video_predictions.append(vp)
        for group_col in ['attack_type','quality']:
            for group,df in scored.groupby(group_col):
                subgroup_rows.append({'run_id':run_id,'split':split,'group_by':group_col,'group':group,
                  **safe_metrics(df.binary_label.to_numpy(),df.video_score.to_numpy(),threshold,int((cv[group_col]==group).sum()))})
        print(f'[Result] {run_id} / {split}: video AUC={video_metrics["auc"]}, HTER={video_metrics["hter"]}, frame AUC={frame_metrics["auc"]}',flush=True)
    # Save summaries after every completed run so interruption preserves completed results.
    pd.DataFrame(summary).to_csv(OUTPUT_ROOT/'comparison_summary.csv',index=False)
    write_json(OUTPUT_ROOT/'comparison_summary.json',summary)
    write_json(run_output/'completion.json',{'run_id':run_id,'status':'completed','elapsed_seconds':time.monotonic()-started})
    print(f'[Done] {run_id}: train / test / combined saved',flush=True)
    del model,logits,chunks,x,z;gc.collect();torch.cuda.empty_cache()
frame_predictions=pd.concat(all_frame_predictions,ignore_index=True)
video_predictions=pd.concat(all_video_predictions,ignore_index=True)
# All model comparisons are paired on exactly the same ordered localized frames/videos.
keys=[df.image_path.tolist() for df in all_frame_predictions]
assert all(k==keys[0] for k in keys)
frame_predictions.to_csv(OUTPUT_ROOT/'all_frame_predictions.csv',index=False)
video_predictions.to_csv(OUTPUT_ROOT/'all_video_predictions.csv',index=False)
pd.DataFrame(subgroup_rows).to_csv(OUTPUT_ROOT/'video_subgroup_metrics.csv',index=False)
comparison=pd.DataFrame(summary)
display(comparison[['run_id','split','best_epoch','locked_source_threshold','auc','eer','apcer','bpcer','acer','hter','binary_accuracy','tpr_at_fpr_1pct','scored_n','candidate_n','frame_auc','frame_acer']])



[Inference] C_100K_crop15: 4063 shared crops, 16 batches; threshold=0.1822357177734375
[Inference] C_100K_crop15 batch 1/16; elapsed=0.0s
[Inference] C_100K_crop15 batch 5/16; elapsed=0.2s
[Inference] C_100K_crop15 batch 10/16; elapsed=0.3s
[Inference] C_100K_crop15 batch 15/16; elapsed=0.5s
[Inference] C_100K_crop15 batch 16/16; elapsed=1.9s
[Result] C_100K_crop15 / train: video AUC=0.8777375737954676, HTER=0.2257189106836793, frame AUC=0.8756054562290762
[Result] C_100K_crop15 / test: video AUC=0.8542527548209367, HTER=0.25757575757575757, frame AUC=0.8282339104174059
[Result] C_100K_crop15 / combined: video AUC=0.8651741293532338, HTER=0.25219734660033166, frame AUC=0.8433634850481562
[Done] C_100K_crop15: train / test / combined saved

[Inference] P3_SF_100K_crop15: 4063 shared crops, 16 batches; threshold=0.29180908203125
[Inference] P3_SF_100K_crop15 batch 1/16; elapsed=0.0s
[Inference] P3_SF_100K_crop15 batch 5/16; elapsed=0.2s
[Inference] P3_SF_100K_crop15 batch 10/16; elapsed

,run_id,split,best_epoch,locked_source_threshold,auc,eer,apcer,bpcer,acer,hter,binary_accuracy,tpr_at_fpr_1pct,scored_n,candidate_n,frame_auc,frame_acer
0,C_100K_crop15,train,15,0.182236,0.877738,0.219720,0.095506,0.355932,0.225719,0.225719,0.839662,0.338983,237,240,0.875605,0.208660
1,C_100K_crop15,test,15,0.182236,0.854253,0.238636,0.094697,0.420455,0.257576,0.257576,0.823864,0.340909,352,360,0.828234,0.278222
2,C_100K_crop15,combined,15,0.182236,0.865174,0.228648,0.093284,0.411111,0.252197,0.252197,0.826816,0.355556,358,360,0.843363,0.255235
3,P3_SF_100K_crop15,train,15,0.291809,0.887355,0.239431,0.095506,0.305085,0.200295,0.200295,0.852321,0.305085,237,240,0.887202,0.198128
4,P3_SF_100K_crop15,test,15,0.291809,0.861355,0.257576,0.106061,0.363636,0.234848,0.234848,0.829545,0.397727,352,360,0.837394,0.268333
5,P3_SF_100K_crop15,combined,15,0.291809,0.871434,0.245357,0.100746,0.355556,0.228151,0.228151,0.835196,0.388889,358,360,0.851307,0.249542
6,R7_SC_100K_crop15,train,11,-0.373016,0.898876,0.219720,0.044944,0.440678,0.242811,0.242811,0.856540,0.288136,237,240,0.891508,0.231993
7,R7_SC_100K_crop15,test,11,-0.373016,0.900654,0.179924,0.049242,0.443182,0.246212,0.246212,0.852273,0.443182,352,360,0.874368,0.255780
8,R7_SC_100K_crop15,combined,11,-0.373016,0.908126,0.178441,0.044776,0.433333,0.239055,0.239055,0.857542,0.455556,358,360,0.882675,0.243187
9,C_FULL_crop15,train,9,5.130859,0.920967,0.202819,0.005618,0.508475,0.257046,0.257046,0.869198,0.491525,237,240,0.914860,0.231772


In [16]:
# Report, ROC figures, and downloadable ZIP. No target-calibrated operating point.
for scale in ['100K','full-scale']:
    ids=[info['run_id'] for info in provenance if info['scale']==scale]
    if not ids:continue
    for split in POOLS:
        fig,ax=plt.subplots(figsize=(7,6))
        for run_id in ids:
            df=video_predictions.loc[(video_predictions.run_id==run_id)&(video_predictions.split==split)&video_predictions.scorable]
            if set(df.binary_label)!={0,1}:continue
            fpr,tpr,_=roc_curve((df.binary_label==0).astype(int),df.video_score)
            ax.plot(fpr,tpr,label=LABELS[run_id])
        ax.plot([0,1],[0,1],'k--',alpha=.5)
        ax.set(xlabel='Attack acceptance rate (FPR)',ylabel='Real acceptance rate (TPR)',title=f'CASIA Kaggle image copy — {scale} / {split}, video ROC')
        handles,_=ax.get_legend_handles_labels()
        if handles:ax.legend()
        fig.tight_layout();fig.savefig(OUTPUT_ROOT/f'roc_video_{scale}_{split}.png',dpi=200);plt.close(fig)
lines=['# CASIA-FASD Kaggle train/test image-copy evaluation','',
 'Evaluation only. No CASIA training, adaptation, threshold fitting or checkpoint selection.', '',
 'All available train and test color JPEGs were attempted; detection/crop failures remain in coverage CSVs. No GT/full-image bbox fallback.',
 'The six checkpoints share the same ordered SCRFD detections, crop1.5 and 80×80 BGR tensors. Spectral views and auxiliary heads are training-only.', '',
 '## Dataset and metric units','',
 f'Raw train/test candidate frames: {len(frame_manifest)}; valid raw frames: {len(valid)}. Populations: {populations}.',
 'Train and test are evaluated separately. Combined is the unique union of reconstructed frame IDs, preferring test copies before detection; it does not count overlapping frame IDs twice.',
 'Combined merges unique frames of the same reconstructed video and averages their valid-frame scores. The two split-wise video rows are not averaged or counted twice.',
 f'Primary metrics: mean valid-frame binary-logit score per reconstructed video; at least {MIN_VALID_FRAMES} valid frames. Secondary metrics: all valid frames.',
 'The frozen source threshold is transferred unchanged to both units; video aggregation may change score distributions. No threshold is recalibrated for video scores.',
 'AUC/EER/TPR@FPR1% describe the ROC. APCER/BPCER/ACER/HTER/accuracy use only the frozen source threshold. EER does not set the prediction threshold.',
 'Metrics are fractions (multiply by 100 for percentages); ACER = HTER = (APCER+BPCER)/2 for the pooled binary attack class.',
 'AUC/EER/ACER/HTER need both classes; single-class subsets still report their applicable APCER or BPCER and accuracy. Undefined metrics remain null.', '',
 '## Limitations','',
 'These are extracted-image Kaggle train and test splits, not a verified official subject-disjoint CASIA benchmark. No original full-video evaluation is claimed.',
 'Filename real/fake labels are cross-checked against video-code mapping from the prior CASIA exploration. No label is guessed from a split folder.',
 f'Train/test image-copy overlap audit: {overlap}. Models were trained on CelebA, not on the CASIA train images.',
 'Face selection is heuristic: largest positive finite SCRFD box, confidence tie-break. Raw bbox records and failures are exported.',
 '100K source selection/calibration: Val15K. Full-scale source selection/calibration: official CelebA Test-as-Val (reused source selection set). Scale groups remain separate.',
 'No statistical significance or best-model selection on CASIA is claimed.', '',
 '## Selected runs and metrics','']
columns=['run_id','split','auc','eer','apcer','bpcer','acer','hter','binary_accuracy','scored_n','candidate_n','frame_auc','frame_acer']
lines += ['| '+' | '.join(columns)+' |','| '+' | '.join(['---']*len(columns))+' |']
for row in summary:
    values=[str(row[c]) if row[c] is not None else 'undefined' for c in columns]
    lines.append('| '+' | '.join(values)+' |')
lines += ['', 'Source provenance and thresholds: `model_provenance.json`; effective settings: `evaluation_protocol.json`.',
          'Per-run frame/video predictions and metrics: `runs/<run_id>/<train|test|combined>/`.',
          'Use comparison_summary.csv/json to merge independent RUN_FILTER notebooks. Check evaluation protocol and coverage match before merging.','']
(OUTPUT_ROOT/'casia_train_test_evaluation_report.md').write_text('\n'.join(lines))
archive_path=OUTPUT_ROOT.with_suffix('.zip')
with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_ROOT.rglob('*')):
        if path.is_file() and path.name!='shared_crops_bgr_uint8.npy' and not path.name.endswith('.partial.csv'):
            archive.write(path,arcname=path.relative_to(OUTPUT_ROOT))
print('Saved report and ZIP:',archive_path,flush=True)
print('Shared crop cache is excluded from the ZIP; raw detection/crop metadata is included.')
display(FileLink(str(archive_path)))


Saved report and ZIP: /kaggle/working/casia_train_test_C_P3SF_R7SC_100k_fullscale.zip
Shared crop cache is excluded from the ZIP; raw detection/crop metadata is included.


/kaggle/working/casia_train_test_C_P3SF_R7SC_100k_fullscale.zip